# S10 · Create the target structure

Creates the schemas and the empty Delta tables in the INTERNAL target catalog, one schema per run. **Structure only — no rows move.** Every table arrives empty.

Runs on compute rather than through the catalog CRUD API because that API returns `202 Accepted` and can create nothing, so a table 'created' that way cannot be verified.

---

*Generated from `engine/dataplane/01_create_structure.py` by `engine/target/stage_notebooks.py`. Regenerate with `snowmig.py build-notebooks`; do not hand-edit — an edit here is overwritten on the next build. Change the source instead.*

In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────
# Edit these, then run the notebook top to bottom. Every value is a
# plain Python literal: `None` means the flag is not passed at all,
# and `True` means a bare switch is passed.
#
# Scope and mode are INPUTS. To migrate less, change a value here --
# never edit the stage logic below to make it cover less.
PARAMS = {
    'source-mode': 'connector',
    'source-config': None,
    'source-catalog': None,
    'target-catalog': None,  # REQUIRED
    'schema': None,
    'mode': 'ddl-plan',
    'reports-dir': None,
}


def _argv(params):
    """PARAMS -> argv. None is omitted; True is a bare switch."""
    argv = []
    for key, value in params.items():
        if value is None or value is False:
            continue
        argv.append(f'--{key}')
        if value is not True:
            argv.extend(str(v) for v in (
                value if isinstance(value, (list, tuple)) else [value]))
    return argv


ARGV = _argv(PARAMS)
print('arguments:', ARGV)

_missing = [k for k in ['target-catalog'] if not PARAMS.get(k)]
if _missing:
    raise ValueError(
        f'set these PARAMS before running: {_missing}')

## Shared source helpers

Inlined from `engine/dataplane/snowmig_source.py` so this notebook runs with nothing else uploaded beside it.

In [ ]:
"""How the migration scripts READ Snowflake from inside AIDP. Two modes.

LIVE-VERIFIED 2026-09-16 on a real cluster (Spark 3.5, AIDP 4.x):

  connector (default)  spark.read.format("aidataplatform") with
                       type=SNOWFLAKE. Talked to the account, read a table
                       (87 rows x 9 cols) and ran a pushdown query
                       (`current_user()` = the service user). Needs NO extra
                       cluster library — the format is built in — and needs
                       NO successful catalog crawl.

  external-catalog     three-part names `catalog.schema.table` against a
                       registered EXTERNAL catalog. Cheaper (no per-read
                       Snowflake login) but it can only see what the CRAWLER
                       has already discovered, and on the validated
                       deployment the crawler failed with
                       "CONNECTOR_0067 ... Login has timed out" while the
                       connector above worked with the same credentials.

So: connector mode is the default because it is the one proven end to end,
and external-catalog mode is kept for a deployment whose crawl succeeds.

The option NAMES are the live-verified raw ones, which differ from the
Python helper's keyword names — `user.name` (not `user`), `database.name`,
`authentication.method` ∈ {Basic, KeyPair}, `private.key.content`. A wrong
name fails loud (`DATA_ACCESS_LAYER_0001 - Required spark option ... was not
provided`), which is how these were established.

Credentials come from a CONFIG FILE, never from arguments: the same rule the
control plane follows. Point --source-config at a JSON file shaped like
snowmig-config.example.yaml (JSON, on the workspace mount).
"""
from __future__ import annotations

import json
import pathlib

__all__ = ["SOURCE_MODES", "SourceConfigError", "SnowflakeSource",
           "load_source_config"]

SOURCE_MODES = ("connector", "external-catalog")

AIDP_FORMAT = "aidataplatform"


class SourceConfigError(ValueError):
    """The source config is missing, unreadable or incomplete."""


def q(identifier: str) -> str:
    """Backtick-quote one Spark identifier."""
    return "`" + str(identifier).replace("`", "``") + "`"


def _sql_ident(identifier: str) -> str:
    """Double-quote one SNOWFLAKE identifier (the pushdown runs there)."""
    return '"' + str(identifier).replace('"', '""') + '"'


def _sql_literal(value: str) -> str:
    """Escape a string literal for Snowflake SQL."""
    return str(value).replace("'", "''")


def load_source_config(path: str | pathlib.Path) -> dict:
    """Read the Snowflake connection config (JSON) from the workspace."""
    p = pathlib.Path(path).expanduser()
    try:
        text = p.read_text()
    except OSError as exc:
        raise SourceConfigError(
            f"source config not readable at {p}: {exc.strerror}") from exc
    if p.suffix.lower() in (".yaml", ".yml"):
        try:
            import yaml
        except ImportError as exc:
            raise SourceConfigError(
                f"{p} is YAML but PyYAML is not on the cluster; write the "
                f"config as JSON instead") from exc
        data = yaml.safe_load(text) or {}
    else:
        data = json.loads(text) if text.strip() else {}
    if not isinstance(data, dict):
        raise SourceConfigError(f"{p}: expected a mapping at the top level")
    # THE MIGRATION CONFIG IS ONE FILE FOR BOTH ENDS: the Snowflake connection
    # nested under `snowflake:`, the AIDP coordinates under `aidp:`. That is
    # the file `provision --source-config` uploads, and it uploads it
    # verbatim -- so the shape that actually reaches the mount is the nested
    # one, and reading the top level for `account` found nothing but the two
    # envelope keys. The failure surfaced on the cluster, as every required
    # field missing at once, which reads like a broken credential rather than
    # a config one level too deep.
    nested = data.get("snowflake")
    if isinstance(nested, dict):
        return dict(nested)
    return data


class SnowflakeSource:
    """Read-only access to the Snowflake source, in either mode.

    Nothing here can write: every method issues a read, and the connector is
    read-only in AIDP 4.0 by Oracle's own statement.
    """

    def __init__(self, spark, *, mode: str = "connector",
                 config: dict | None = None,
                 external_catalog: str | None = None,
                 session_schema: str | None = None):
        if mode not in SOURCE_MODES:
            raise SourceConfigError(
                f"unknown source mode {mode!r}; expected one of "
                f"{list(SOURCE_MODES)}")
        self.spark = spark
        self.mode = mode
        self.external_catalog = external_catalog
        self._options: dict[str, str] = {}
        # A REAL schema, used only to scope a pushdown session. The connector
        # validates this option against the schemas it can see and rejects
        # INFORMATION_SCHEMA itself with DATA_ACCESS_LAYER_0031 -- but a
        # pushdown query scoped to a real schema may reference
        # INFORMATION_SCHEMA freely (both established live).
        self.session_schema = session_schema or (config or {}).get("schema")

        if mode == "external-catalog":
            if not external_catalog:
                raise SourceConfigError(
                    "external-catalog mode needs --source-catalog")
            return

        cfg = config or {}
        missing = [k for k in ("account", "warehouse", "database", "user",
                               "auth") if not cfg.get(k)]
        if missing:
            raise SourceConfigError(
                "source config is missing required field(s): "
                + ", ".join(sorted(missing)))
        account = str(cfg["account"]).strip()
        # The connector wants the account/server URL host, which is what the
        # Snowflake console calls "Account/Server URL".
        host = str(cfg.get("host")
                   or f"{account}.snowflakecomputing.com").strip()
        opts = {
            "type": "SNOWFLAKE",
            "host": host,
            "port": str(cfg.get("port") or 443),
            "database.name": str(cfg["database"]).strip(),
            "user.name": str(cfg["user"]).strip(),
            "warehouse": str(cfg["warehouse"]).strip(),
        }
        if cfg.get("role"):
            opts["role"] = str(cfg["role"]).strip()

        # A secret may be inline in the one config file, or in a file the
        # config points at. On a cluster the inline form is usually the only
        # one available, since the workspace mount carries the config but not
        # the operator's home directory.
        def secret(inline, path_field):
            if cfg.get(inline):
                return str(cfg[inline])
            if cfg.get(path_field):
                return pathlib.Path(
                    str(cfg[path_field])).expanduser().read_text().strip()
            return None

        auth = str(cfg["auth"]).strip().lower()
        if auth == "keypair":
            key = secret("private_key", "key_path")
            if not key:
                raise SourceConfigError(
                    "auth: keypair needs `private_key` (inline) or `key_path`")
            opts["authentication.method"] = "KeyPair"
            opts["private.key.content"] = key
            passphrase = secret("key_passphrase", "key_passphrase_path")
            if passphrase:
                opts["private.key.pass.phrase"] = passphrase
        elif auth == "password":
            password = secret("password", "password_path")
            if not password:
                raise SourceConfigError(
                    "auth: password needs `password` (inline) or "
                    "`password_path`")
            opts["authentication.method"] = "Basic"
            opts["password"] = password
        else:
            raise SourceConfigError(
                f"auth {auth!r} is not supported by the AIDP Snowflake "
                f"connector; use keypair (preferred) or password")
        self._options = opts

    # -- describing the estate ------------------------------------------

    def database(self) -> str | None:
        return self._options.get("database.name")

    def pushdown(self, sql: str, *, schema: str | None = None):
        """Run `sql` IN SNOWFLAKE and return a DataFrame.

        Connector mode only. The `schema` option must name a REAL schema:
        the connector rejects INFORMATION_SCHEMA there with
        DATA_ACCESS_LAYER_0031, while a query scoped to a real schema may
        reference INFORMATION_SCHEMA freely. Both established live.
        """
        if self.mode != "connector":
            raise SourceConfigError(
                "pushdown is connector-mode only; external-catalog mode has "
                "no Snowflake session to push into")
        scope = schema or self.session_schema
        if not scope:
            raise SourceConfigError(
                "connector pushdown needs a REAL schema to scope the "
                "session: add `schema:` to the source config or pass "
                "--session-schema. INFORMATION_SCHEMA is not accepted there.")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", scope)
                .option("pushdown.sql", sql)
                .load())

    def source_counts(self, schema: str, tables: list[str], *,
                      chunk: int = 50) -> dict[str, int]:
        """`{table: COUNT(*)}` for many tables in ONE round trip per chunk.

        Connector mode opens a Snowflake session per read, and the copy needs
        a source count twice per table (before, and again after, to catch a
        source that moved during the copy). Per-table counting therefore cost
        more session setup than the copy itself on a live run. A single
        UNION ALL answers a whole schema instead; it is chunked because a
        statement with thousands of branches is its own problem.

        External-catalog mode has no session to amortise, so it falls back to
        a count per table -- correct either way, and the caller does not care.
        """
        out: dict[str, int] = {}
        if self.mode != "connector":
            for table in tables:
                out[table] = self.read_table(schema, table).count()
            return out
        for start in range(0, len(tables), chunk):
            batch = tables[start:start + chunk]
            sql = " union all ".join(
                # The literal is the table's own name, so one query can carry
                # many counts and still say which is which. Both the literal
                # and the identifier are escaped: one apostrophe in a table
                # name would otherwise break the whole chunk.
                f"select '{_sql_literal(t)}' as SNOWMIG_TABLE, "
                f"count(*) as SNOWMIG_N from {_sql_ident(t)}"
                for t in batch)
            for row in self.pushdown(sql, schema=schema).collect():
                data = row.asDict()
                out[str(data["SNOWMIG_TABLE"])] = int(data["SNOWMIG_N"])
        return out

    def read_table(self, schema: str, table: str):
        """A DataFrame over one source table."""
        if self.mode == "external-catalog":
            return self.spark.table(
                f"{q(self.external_catalog)}.{q(schema)}.{q(table)}")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", schema)
                .option("table", table)
                .load())

    def register_temp_view(self, schema: str, table: str, view: str) -> str:
        """Expose a source table to SQL as a temp view, and return its name.

        INSERT ... SELECT needs the source addressable in SQL. In
        external-catalog mode the three-part name already is; in connector
        mode the DataFrame is registered as a session-local temp view, which
        is dropped by the caller.
        """
        if self.mode == "external-catalog":
            return f"{q(self.external_catalog)}.{q(schema)}.{q(table)}"
        self.read_table(schema, table).createOrReplaceTempView(view)
        return q(view)

    def drop_temp_view(self, view: str) -> None:
        if self.mode == "connector":
            self.spark.catalog.dropTempView(view)

    def describe(self) -> dict:
        """What this source is, for the report header. Never the credential."""
        out = {"mode": self.mode}
        if self.mode == "external-catalog":
            out["external_catalog"] = self.external_catalog
        else:
            out.update(session_schema=self.session_schema,
                       host=self._options.get("host"),
                       database=self._options.get("database.name"),
                       user=self._options.get("user.name"),
                       warehouse=self._options.get("warehouse"),
                       role=self._options.get("role"),
                       auth=self._options.get("authentication.method"))
        return out


## Stage logic

In [ ]:
#!/usr/bin/env python3
"""Create target schemas and EMPTY Delta tables for one schema (or all).

Runs on AIDP compute. Three structure sources, chosen with --mode:

  ddl-plan (default)  types come from `plan/ddl_plan.json`, which the
                      migrator's own type mapper produced -- it refuses what
                      it cannot map exactly instead of guessing, and the plan
                      was reviewed and signed off before this ran. Costs NO
                      source read per table, which is what makes a large
                      estate feasible: creating 100 tables by CTAS took
                      minutes on a live run, because each CTAS is its own
                      Snowflake round trip.
  ctas                CREATE TABLE ... USING DELTA AS SELECT * ... WHERE 1=0.
                      Spark derives the types through the connector, so the
                      copy cannot hit a type the table cannot hold -- but the
                      mapping is the connector's, not an audited one, and it
                      pays a source read per table.
  manifest            types come from discovery_manifest.json verbatim. Only
                      valid when the manifest carries SPARK types, i.e. it
                      was built in external-catalog mode (DESCRIBE); a
                      connector-mode manifest carries SNOWFLAKE types and is
                      refused here rather than mistranslated.

Safety: CREATE TABLE IF NOT EXISTS everywhere; nothing is ever dropped or
replaced here. The target catalog must be INTERNAL — this script REFUSES to
address the source catalog as its target, and AIDP refuses DDL on external
catalogs anyway (documented), so the failure would be loud, not silent.

Writes `structure_report_<schema>.json` per schema: created / already-existed
/ failed, each with a reason. Resumable: an object already recorded as created
is skipped (--force re-issues the IF NOT EXISTS create, which is idempotent).
"""
from __future__ import annotations

import argparse
import datetime
import json
import pathlib
import sys


# /Workspace is the live-verified mount of the workspace tree on cluster
# filesystems (probed 2026-09-16 on a real cluster).
DEFAULT_REPORTS_DIR = "/Workspace/backup-snowflake-migration/reports"
MANIFEST_NAME = "discovery_manifest.json"


def three(*parts: str) -> str:
    return ".".join(q(p) for p in parts)


def log(msg: str) -> None:
    print(f"[structure] {msg}", flush=True)


def fail(msg: str) -> int:
    """Report a refusal on BOTH streams and return 1.

    A notebook task captures stdout only: live, a script that exited 1 via a
    stderr-only message produced a job failure with NO explanation anywhere.
    """
    print(f"ERROR: {msg}", flush=True)
    print(f"error: {msg}", file=sys.stderr)
    return 1


def _report_path(reports: pathlib.Path, schema: str) -> pathlib.Path:
    return reports / f"structure_report_{schema.lower()}.json"


def _load_report(path: pathlib.Path, schema: str, target: str) -> dict:
    """The prior report for this schema, but ONLY if it targeted the same place.

    Resumability is keyed by SOURCE schema, so a report written while
    targeting one destination would otherwise let a run against a DIFFERENT
    destination skip every create as "already created" -- observed live, with
    an empty target schema reported as done. A changed target starts a fresh
    record and says so.
    """
    if not path.exists():
        return {"schema": schema, "objects": {}, "target": target}
    prior = json.loads(path.read_text())
    if prior.get("target") and prior["target"] != target:
        log(f"{schema}: the previous report targeted {prior['target']}, not "
            f"{target} — starting a fresh record for this target (the old "
            f"one is kept at {path.name}.{prior['target'].replace('.', '_')})")
        path.with_suffix(
            f".{prior['target'].replace('.', '_')}.json").write_text(
                json.dumps(prior, indent=2))
        return {"schema": schema, "objects": {}, "target": target}
    return prior


def create_table_ctas(source: SnowflakeSource, schema: str, name: str,
                      target_catalog: str, target_schema: str) -> None:
    """Empty table whose columns Spark derives from the SOURCE read.

    The source is addressed through `SnowflakeSource`, so this works in
    connector mode (a temp view over the connector read) as well as against
    an external catalog's three-part name.
    """
    view = f"snowmig_src_{schema}_{name}".lower()[:120]
    ref = source.register_temp_view(schema, name, view)
    try:
        source.spark.sql(
            f"CREATE TABLE IF NOT EXISTS "
            f"{three(target_catalog, target_schema, name)} "
            f"USING DELTA AS SELECT * FROM {ref} WHERE 1=0")
    finally:
        source.drop_temp_view(view)


def create_table_from_columns(spark, columns: list[dict],
                              target_catalog: str, target_schema: str,
                              name: str) -> None:
    """CREATE TABLE from an explicit column list. Types are used verbatim."""
    if not columns:
        raise ValueError("no column list for this table; rediscover it or "
                         "use --mode ctas")
    cols = ", ".join(f'{q(c["name"])} {c["type"]}' for c in columns)
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {three(target_catalog, target_schema, name)} "
        f"({cols}) USING DELTA")


# Types Snowflake reports but Spark/Delta does not accept verbatim. Their
# presence in a manifest means it was built in CONNECTOR mode, where the
# manifest records SOURCE types on purpose -- translating them here would
# duplicate (and inevitably diverge from) the migrator's own type mapper,
# which refuses ambiguous cases rather than guessing.
_SNOWFLAKE_ONLY_TYPES = ("NUMBER", "TEXT", "VARIANT", "OBJECT", "GEOGRAPHY",
                         "GEOMETRY", "TIMESTAMP_LTZ", "TIMESTAMP_TZ")


def _looks_like_snowflake_types(columns: list[dict]) -> bool:
    return any(str(c.get("type", "")).upper().startswith(t)
               for c in columns for t in _SNOWFLAKE_ONLY_TYPES)


def columns_from_ddl_plan(ddl_plan: dict) -> dict[tuple[str, str], list[dict]]:
    """{(source_schema, table): [{name, type}]} from the engine's ddl_plan.

    The engine translated these types with its full discipline (it blocks a
    table it cannot map exactly), and the plan was the artifact the user
    signed off, so applying it needs no source read and no cluster-side
    translation.
    """
    out: dict[tuple[str, str], list[dict]] = {}
    for stmt in ddl_plan.get("statements") or []:
        ident = str(stmt.get("source_identifier") or "")
        parts = ident.split(".")
        if len(parts) != 3 or not stmt.get("expected_columns"):
            continue
        out[(parts[1], parts[2])] = stmt["expected_columns"]
    return out


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--source-mode", choices=list(SOURCE_MODES),
                    default="connector",
                    help="how to READ the source when --mode ctas derives "
                         "types from it (see snowmig_source.py)")
    ap.add_argument("--source-config",
                    help="JSON/YAML connection config (connector mode)")
    ap.add_argument("--source-catalog",
                    help="the registered EXTERNAL catalog "
                         "(external-catalog mode)")
    ap.add_argument("--target-catalog", required=True)
    ap.add_argument("--schema", action="append", default=None,
                    help="repeatable; default: every schema in the manifest")
    ap.add_argument("--target-schema", default=None,
                    help="override the target schema name (single --schema "
                         "runs only); default mirrors the source")
    ap.add_argument("--mode", choices=("ddl-plan", "ctas", "manifest"),
                    default="ddl-plan")
    ap.add_argument("--ddl-plan",
                    help="path to ddl_plan.json (default: ../plan/"
                         "ddl_plan.json next to --reports-dir)")
    ap.add_argument("--reports-dir", default=DEFAULT_REPORTS_DIR)
    ap.add_argument("--dry-run", action="store_true",
                    help="print every statement; execute nothing")
    ap.add_argument("--force", action="store_true",
                    help="re-issue creates the report already records")
    args = ap.parse_args(argv)

    if args.source_catalog and \
            args.source_catalog.lower() == args.target_catalog.lower():
        return fail("error: source and target catalog are the same. The source is "
              "the read-only EXTERNAL catalog; the target must be an INTERNAL "
              "one.")
    if args.target_schema and len(args.schema or []) != 1:
        return fail("error: --target-schema needs exactly one --schema")

    reports = pathlib.Path(args.reports_dir)
    manifest = json.loads((reports / MANIFEST_NAME).read_text())
    by_name = {s["name"]: s for s in manifest["schemas"]}
    schemas = args.schema or sorted(by_name)

    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    planned_columns: dict = {}
    if args.mode == "ddl-plan":
        ddl_path = (pathlib.Path(args.ddl_plan) if args.ddl_plan
                    else reports.parent / "plan" / "ddl_plan.json")
        if not ddl_path.is_file():
            return fail(f"--mode ddl-plan needs ddl_plan.json; {ddl_path} is "
                        f"not there. Run the migrator's `ddl` stage and let "
                        f"`provision` upload it, or pass --ddl-plan")
        planned_columns = columns_from_ddl_plan(
            json.loads(ddl_path.read_text()))
        log(f"ddl plan: {len(planned_columns)} table(s) with engine-"
            f"translated types, from {ddl_path}")

    source = None
    if args.mode == "ctas":
        try:
            config = (load_source_config(args.source_config)
                      if args.source_config else None)
            source = SnowflakeSource(spark, mode=args.source_mode,
                                     config=config,
                                     external_catalog=args.source_catalog)
        except SourceConfigError as exc:
            return fail(f"error: {exc}")

    failures = 0
    for schema in schemas:
        record = by_name.get(schema)
        if record is None:
            return fail(f"error: schema {schema!r} is not in the manifest; run "
                  f"00_discover first")
        target_schema = args.target_schema or schema
        path = _report_path(reports, schema)
        target = f"{args.target_catalog}.{target_schema}"
        report = _load_report(path, schema, target)
        report["target"] = target
        report["mode"] = args.mode

        schema_sql = (f"CREATE SCHEMA IF NOT EXISTS "
                      f"{q(args.target_catalog)}.{q(target_schema)}")
        if args.dry_run:
            log(f"DRY RUN: {schema_sql}")
        else:
            spark.sql(schema_sql)

        for table in record["tables"]:
            name = table["name"]
            prior = report["objects"].get(name, {})
            if prior.get("status") == "created" and not args.force:
                log(f"skip {schema}.{name}: already created")
                continue
            try:
                if args.dry_run:
                    log(f"DRY RUN: would create "
                        f"{args.target_catalog}.{target_schema}.{name} "
                        f"({args.mode})")
                    status = "dry_run"
                elif args.mode == "ctas":
                    create_table_ctas(source, schema, name,
                                      args.target_catalog, target_schema)
                    status = "created"
                elif args.mode == "ddl-plan":
                    columns = planned_columns.get((schema, name))
                    if not columns:
                        report["objects"][name] = {
                            "status": "not_in_plan",
                            "reason": "the approved ddl_plan carries no "
                                      "columns for this table -- the engine "
                                      "either blocked it or it was outside "
                                      "the plan's scope. NOT created."}
                        path.write_text(json.dumps(report, indent=2))
                        log(f"{schema}.{name}: not in the approved plan")
                        continue
                    create_table_from_columns(spark, columns,
                                              args.target_catalog,
                                              target_schema, name)
                    status = "created"
                else:
                    columns = table.get("columns") or []
                    if _looks_like_snowflake_types(columns):
                        raise ValueError(
                            "this manifest carries SNOWFLAKE types (it was "
                            "built in connector mode), which Delta will not "
                            "accept verbatim. Use --mode ddl-plan (engine-"
                            "translated types) or --mode ctas.")
                    create_table_from_columns(spark, columns,
                                              args.target_catalog,
                                              target_schema, name)
                    status = "created"
                report["objects"][name] = {"status": status}
                log(f"{schema}.{name}: {status}")
            except Exception as exc:
                failures += 1
                report["objects"][name] = {"status": "failed",
                                           "reason": str(exc)[:400]}
                log(f"{schema}.{name}: FAILED — {str(exc)[:200]}")
            report["updated_at"] = datetime.datetime.now(
                datetime.timezone.utc).isoformat()
            path.write_text(json.dumps(report, indent=2))

        counts = {}
        for obj in report["objects"].values():
            counts[obj["status"]] = counts.get(obj["status"], 0) + 1
        log(f"{schema}: {counts} -> {path}")

    return 1 if failures else 0



In [ ]:
# ── RUN ────────────────────────────────────────────────────────────
# main() RETURNS an exit code; it is not allowed to raise SystemExit here.
# A notebook cell that raises SystemExit is reported as a FAILED task even
# when the work succeeded, so the code is inspected and only a real failure
# is re-raised -- which keeps a genuinely failed stage failing.
code = main(ARGV)
print('exit code:', code, flush=True)
if code:
    raise RuntimeError(f'stage exited {code}')
